# 03 — Detect fraud (blended CU reasoning + rule engine)

Runs the `proFraudV1` and `proClaimsV1` analyzers over the `claim_auto_collision_fraud` sample (same FNOL/police/photo as the clean case, but with a tampered repair estimate). The CU classify findings feed a small rule engine; the two are blended into a 0–100 risk score.

In [ ]:
from _lib import pro_service, fraud_rules, SAMPLES_DIR, CU_OUTPUT_DIR
import json

manifest, files = pro_service.load_sample('claim_auto_collision_fraud')
print('Sample :', manifest.id)
for p in files:
    flag = ' (tampered)' if any(f.name == p.name and f.tampered for f in manifest.files) else ''
    print(f'  - {p.name:25s} ({p.stat().st_size:,} bytes){flag}')

In [ ]:
result = pro_service.analyze_fraud(files, sample_id='claim_auto_collision_fraud')
print(f'Risk score : {result.risk_score}/100  band={result.risk_band.upper()}')
print(f'CU signals : {len(result.cu_signals)}')
print(f'Rule signals: {len(result.rule_signals)}')
print()
for s in [*result.rule_signals, *result.cu_signals]:
    print(f'  [{s.severity:6s}] {s.rule_id}  (w={s.weight})')
    print(f'      {s.evidence}')

In [ ]:
print('Overall CU rating:', result.fields.overall_fraud_indication)
print()
print(result.fields.rationale or '(no rationale)')

In [ ]:
# Persist combined raw response
out = CU_OUTPUT_DIR / 'claim_auto_collision_fraud.json'
out.write_text(json.dumps(result.raw, indent=2, default=str), encoding='utf-8')
print('Saved:', out)

**Expected signals**: `TOTALS_EXCEED_SUBLIMIT` (rule, high), `DATE_IMPLAUSIBLE` (rule, high), `VIN_MISMATCH` (rule, medium), plus mirroring CU-reasoning signals. Risk score should land in the 70–95 band.